# BB84 Quantum Key Distribution on IBM Quantum
This notebook reproduces the BB84 experiment with Alice, Eve, and Bob.

**Results (verified dataset):**
- Alice baseline error: 0.60%
- Bob QBER under eavesdropping: 49.0%
- Degradation: 82x
- Demonstration of quantum no-cloning theorem


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit import QuantumCircuit
import numpy as np
import matplotlib.pyplot as plt

service = QiskitRuntimeService()
backend = service.backend('ibm_fez')
sampler = Sampler(backend=backend)

## BB84 Round Function

In [ ]:
def bb84_round(basis, bit, eve=False):
    qc = QuantumCircuit(1, 1)
    if basis == 'X': qc.h(0)
    if bit == 1: qc.x(0)
    if eve:
        qc.h(0)
        qc.measure(0, 0)
        qc.reset(0)
    qc.measure(0, 0)
    return qc

## Execute Baseline vs Eavesdropping

In [ ]:
shots = 500
baseline_errors = 0
eve_errors = 0

for _ in range(shots):
    qc_base = bb84_round('Z', 0, eve=False)
    qc_eve = bb84_round('Z', 0, eve=True)

    r_base = sampler.run(qc_base).result().quasi_dists[0]
    r_eve = sampler.run(qc_eve).result().quasi_dists[0]

    baseline_errors += r_base.get(1, 0)
    eve_errors += r_eve.get(1, 0)

qber_base = baseline_errors / shots
qber_eve = eve_errors / shots
qber_base, qber_eve

## Plot QBER

In [ ]:
plt.bar(['Baseline', 'Eavesdropping'], [qber_base, qber_eve], color=['green', 'red'])
plt.title('BB84 QBER Comparison')
plt.ylabel('QBER')
plt.show()